In [1]:
from datasets import load_dataset
import numpy as np
import tensorflow as tf
from PIL import Image

In [7]:
imagenet_int8 = load_dataset("theodor1289/imagenet-1k_tiny")

README.md: 0.00B [00:00, ?B/s]

C:\Users\jackz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jackz\.cache\huggingface\hub\datasets--theodor1289--imagenet-1k_tiny. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


(…)-00000-of-00001-b6bc71b3cb8f3afd.parquet:   0%|          | 0.00/11.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100 [00:00<?, ? examples/s]

In [38]:
# load our model
interpreter = tf.lite.Interpreter(model_path="model/smallmodel.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [39]:
#choose image to test on:
img = imagenet_int8["train"][1]['image']
label = imagenet_int8["train"][1]['label']


#resize input to input size
input_shape = input_details[0]['shape']
target_size = (input_shape[2], input_shape[1])  
img = img.resize(target_size)
print(f"Resizing input image to {target_size}")

Resizing input image to (np.int32(224), np.int32(224))


In [27]:
#add batch dimension
input_data = np.expand_dims(img, axis=0).astype(input_details[0]['dtype'])
interpreter.set_tensor(input_details[0]['index'], input_data)

In [28]:
#INFERENCE
interpreter.invoke()

In [30]:
#get output
output_data = interpreter.get_tensor(output_details[0]['index'])
#get output class
predicted_class = np.argmax(output_data)
print("Predicted class:", predicted_class)

Predicted class: 611


In [49]:
def run_inference(intepreter, img) -> int:
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_shape = input_details[0]['shape']
    target_size = (input_shape[2], input_shape[1])  
    img = img.resize(target_size)

    #add batch dimension
    input_data = np.expand_dims(img, axis=0).astype(input_details[0]['dtype'])
    if np.issubdtype(input_details[0]['dtype'], np.floating):
        input_data = input_data / 255.0
    interpreter.set_tensor(input_details[0]['index'], input_data)

    #INFERENCE
    interpreter.invoke()

    #get output
    output_data = interpreter.get_tensor(output_details[0]['index'])
    #get output class
    predicted_class = np.argmax(output_data)

    return predicted_class

In [51]:
# load our model
interpreter = tf.lite.Interpreter(model_path="model/smallmodel.tflite")
interpreter.allocate_tensors()

top_1 = 0

for i in range(len(imagenet_int8["train"])):
    img = imagenet_int8["train"][i]["image"]
    expected = imagenet_int8["train"][i]["label"]

    try:
        inferred = run_inference(interpreter, img)
        top_1 += inferred == expected
    except:
        pass


#accuracy
accuracy = (top_1 / len(imagenet_int8["train"])) * 100
print(f"Accuracy: {accuracy}%")


Accuracy: 75.0%
